In [27]:
import numpy as np
dataog = [
    [150, 7.0, 1, 'Apple'],
    [120, 6.5, 0, 'Banana'],
    [180, 7.5, 2, 'Orange'],
    [155, 7.2, 1, 'Apple'],
    [110, 6.0, 0, 'Banana'],
    [190, 7.8, 2, 'Orange'],
    [145, 7.1, 1, 'Apple'],
    [115, 6.3, 0, 'Banana']
]
data = np.array(dataog)
#Goal => encode apple=0, banana=1, orange=2 [one-hot encoding]

In [28]:
#Separate data into 'X' and 'Y' where Y is labels, and X is the feature vector matrix
X = data[:,0:3]
Y = data[:,3]
Y[Y=='Apple']=0
Y[Y=='Banana']=1
Y[Y=='Orange']=2
X=np.array(X, dtype=float)
Y=np.array(Y, dtype=int)

In [29]:
#Min-Max / Z-score Normalization
def normalize(X):
    X_norm_mnmx = (X-X.min(axis=0)) / (X.max(axis=0)-X.min(axis=0))
    X_norm_z = (X-X.mean(axis=0))/X.std(axis=0)
    return X_norm_z

#Distance Metrics
def dis(a,b):
    euclidean = np.sqrt(np.sum((a-b)**2,axis=1))
    manhattan = np.sum(np.abs(a-b),axis=1)
    p = 3
    minkowsky = np.sum(np.abs(a-b)**p,axis=1)**(1.0/p)
    return euclidean #Axis=1 refers to the rows

#Test-Train Split
def train_test_split(X,Y,test_size=0.2):
    n=len(X)
    indices = np.arange(n)
    np.random.shuffle(indices)
    split = int(n*(1-test_size)) #This split is for the train data's last index

    train_idx = indices[:split]
    test_idx = indices[split:]
    return train_idx,test_idx

X=normalize(X)

In [30]:
class KNN:
    def __init__(self,k=3):
        self.k=k
        self.X_f=None
        self.y_f=None

    def fit(self,X,Y):
        self.X_f = normalize(np.array(X, dtype=float))
        self.y_f = np.array(Y, dtype=int)

    def predict_one(self,x):
        #We get distances from the fitted data ( X_f )
        distances = dis(self.X_f,x)

        #Now we sort and get k best
        idx = np.argsort(distances)[:self.k]
        k_best = self.y_f[idx]

        #We add weights ( closer is better )
        k_dist = distances[idx]
        weights = 1/(k_dist + 1e-8) # we can't just keep k_dist, we need an epsilon of small value to avoid error

        #now to get our best label
        return np.argmax(np.bincount(k_best,weights=weights))

    def predict(self,X_test):
        predictions = []
        X_test = normalize(X_test)
        for x in X_test:
            predictions.append(self.predict_one(x))
        return predictions

In [35]:
test_data = np.array([
    [118, 6.2, 0],  # Expected: Banana
    [160, 7.3, 1],  # Expected: Apple
    [185, 7.7, 2]   # Expected: Orange
])
model = KNN(k=5)
model.fit(X,Y)

#We need to normalize test_data accordingly
mean = X.mean(axis=0)
std = X.std(axis=0)
test_data = (test_data-mean)/std
predictions = model.predict(test_data)
print(predictions)

#Converting to actual labels
mapping = {0: 'Apple', 1: 'Banana', 2: 'Orange'}
predictions = [mapping[p] for p in predictions]

print(predictions) #It works ^_^

[np.int64(1), np.int64(0), np.int64(2)]
['Banana', 'Apple', 'Orange']


In [36]:
#Simple Accuracy checker
y_test = np.array([1,0,2],dtype=int)
y_pred = model.predict(test_data)

def accuracy_checker(y_test,y_pred):
    return (y_test==y_pred).mean()

print(accuracy_checker(y_test,y_pred))
#Weighted KNN is goated
#Without it k=4,5 were giving labels and accuracy had dropped to 0.6666


1.0


Task Completed Fully\
\
Bonus Challenges completed :\
i) Implement Accuracy Checker\
ii) Normalization using min-max and z-score\
iii) Different distance metrics\
iv) Test-Train Split function created but not used as very less data\
v) Weighted kNN implemented\
\
Incomplete Challenges :\
i)Visualizing in 2D